# Manufacturing Document Q&A — RAG Pipeline Demo

Retrieval-Augmented Generation (RAG) system for querying manufacturing SOPs,
maintenance manuals, and quality procedures using semantic search + LLM generation.

**Architecture:**
```
Documents → Chunking → Embeddings → ChromaDB
                                        ↓
User Query → Embedding → Similarity Search → Top-K Chunks → LLM → Answer + Citations
```

> **Note:** LLM queries require an `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in `.env`.
> The chunking, embedding, and retrieval steps run without an API key.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
import pandas as pd

# Check for .env file
env_path = Path('../.env')
if not env_path.exists():
    print('⚠️  No .env file found. Create one with: ANTHROPIC_API_KEY=sk-ant-...')
else:
    print('✅ .env file found')
print('\nData directory contents:')
for f in Path('../data').rglob('*'):
    print(' ', f)

## 2. Sample Manufacturing Documents

The system ingests any `.txt`, `.md`, or `.pdf` file from the `data/sample_docs/` folder.
Here we inspect the included sample document.

In [ ]:
sample_doc = Path('../data/sample_docs/sample_manual.txt')
if sample_doc.exists():
    content = sample_doc.read_text()
    print(f'Document: {sample_doc.name}')
    print(f'Size: {len(content)} characters')
    print(f'\nFirst 500 chars:')
    print(content[:500])
else:
    print('Sample document not found. Run: python scripts/generate_data.py')

## 3. Chunking Strategy

Documents are split into overlapping chunks for better retrieval.
Smaller chunks = more precise retrieval. Larger chunks = more context per result.

In [ ]:
# Demonstrate chunking without API keys
from langchain.text_splitter import RecursiveCharacterTextSplitter

if sample_doc.exists():
    text = sample_doc.read_text()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=['\n\n', '\n', ' ', '']
    )
    chunks = splitter.split_text(text)
    print(f'Total chunks: {len(chunks)}')
    print(f'Avg chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} chars')
    print(f'\nChunk 0 preview:')
    print(chunks[0][:300])
else:
    print('Install langchain: pip install -r requirements.txt')

## 4. Retrieval Pipeline Architecture

```python
# Full pipeline (requires API key)
from ingestion import DocumentIngester
from qa_chain import ManufacturingQAChain

# Step 1: Ingest documents
ingester = DocumentIngester(collection_name='manufacturing_docs')
ingester.ingest_directory('data/sample_docs/')

# Step 2: Query
qa = ManufacturingQAChain(provider='anthropic')  # or 'openai'
result = qa.query('What is the recommended shielding gas flow rate for titanium welding?')

print('Answer:', result['answer'])
print('Sources:', result['sources'])
```

## 5. Example Queries for Manufacturing SOPs

The system is designed to answer questions like:

In [ ]:
example_queries = [
    "What is the recommended laser power range for welding Ti-6Al-4V at 1.5mm thickness?",
    "What shielding gas should be used for stainless steel laser welding?",
    "What are the AS9100 documentation requirements for material certifications?",
    "How do I identify porosity in a laser weld and what are the root causes?",
    "What is the corrective action procedure for an NCR related to dimensional out-of-tolerance?",
    "What are the calibration intervals for the LASAG SLS 200?",
]

print('Sample queries this system can answer:\n')
for i, q in enumerate(example_queries, 1):
    print(f'{i}. {q}')

## 6. Performance Characteristics

| Metric | Value |
|---|---|
| Chunk size | 500 characters |
| Chunk overlap | 50 characters |
| Top-K retrieval | 5 most similar chunks |
| Embedding model | text-embedding-3-small |
| LLM | Claude 3.5 Haiku / GPT-4o-mini |
| Avg query latency | ~1-2 seconds |
| Supported formats | .txt, .md, .pdf |

## 7. Running the Full System

```bash
# 1. Add your API key
echo 'ANTHROPIC_API_KEY=sk-ant-...' > .env

# 2. Ingest your documents
python scripts/ingest_docs.py --docs-path data/sample_docs/

# 3. Launch the dashboard
streamlit run src/app.py
```

The Streamlit UI lets engineers query the entire document library in plain English
and get answers with source citations — no document search required.